In [ ]:
import cobra
from dGbyG.api import predict_transformed_dG_prime_for_GEM

In [11]:
print(f"Loaded model: {gem.id}")
print(f"Number of reactions: {len(gem.reactions)}")
print(f"Number of metabolites: {len(gem.metabolites)}")

Loaded model: yeastGEM_v9__46__1__46__0
Number of reactions: 4102
Number of metabolites: 2748


In [2]:
gem_path = "./yeast-GEM.xml"
gem = cobra.io.read_sbml_model(gem_path)


Set parameter Username
Academic license - for non-commercial use only - expires 2027-01-06


In [3]:
gem.compartments

{'ce': 'cell envelope',
 'c': 'cytoplasm',
 'e': 'extracellular',
 'm': 'mitochondrion',
 'n': 'nucleus',
 'p': 'peroxisome',
 'er': 'endoplasmic reticulum',
 'g': 'Golgi',
 'lp': 'lipid particle',
 'v': 'vacuole',
 'erm': 'endoplasmic reticulum membrane',
 'vm': 'vacuolar membrane',
 'gm': 'Golgi membrane',
 'mm': 'mitochondrial membrane'}

In [4]:
compartment_conditions = {
    "c":  {"pH": 7.2,  "e_potential": 0.0,    "T": 298.15, "I": 0.25, "pMg": 14.0},
    "e":  {"pH": 7.0,  "e_potential": 0.0,    "T": 298.15, "I": 0.25, "pMg": 14.0},
    "g":  {"pH": 6.35, "e_potential": 0.0,    "T": 298.15, "I": 0.25, "pMg": 14.0},
    "m":  {"pH": 7.5,  "e_potential": -0.155, "T": 298.15, "I": 0.25, "pMg": 14.0},
    "n":  {"pH": 7.2,  "e_potential": 0.0,    "T": 298.15, "I": 0.25, "pMg": 14.0},
    "v":  {"pH": 6.2,  "e_potential": 0.0,    "T": 298.15, "I": 0.25, "pMg": 14.0},
    "ce": {"pH": 7.0,  "e_potential": 0.0,    "T": 298.15, "I": 0.25, "pMg": 14.0},
    "p":  {"pH": 7.4,  "e_potential": 0.0,    "T": 298.15, "I": 0.25, "pMg": 14.0},
    "er": {"pH": 7.2,  "e_potential": 0.0,    "T": 298.15, "I": 0.25, "pMg": 14.0},
    "lp": {"pH": 7.0,  "e_potential": 0.0,    "T": 298.15, "I": 0.25, "pMg": 14.0},
    "erm": {"pH": 7.2, "e_potential": 0.0,    "T": 298.15, "I": 0.25, "pMg": 14.0},
    "vm": {"pH": 6.2,  "e_potential": 0.0,    "T": 298.15, "I": 0.25, "pMg": 14.0},
    "gm": {"pH": 6.35, "e_potential": 0.0,    "T": 298.15, "I": 0.25, "pMg": 14.0},
    "mm": {"pH": 7.5,  "e_potential": -0.155, "T": 298.15, "I": 0.25, "pMg": 14.0},
}


In [5]:
import json
compartments_json_path = "./Yeast9_compartment_conditions.json"
with open(compartments_json_path, "w") as f:
    json.dump(compartment_conditions, f, indent=2)
print(f"Saved compartment conditions to: {compartments_json_path}")



Saved compartment conditions to: ./Yeast9_compartment_conditions.json


In [6]:
cid_types = sorted({
    cid_type
    for met in gem.metabolites
    for cid_type in met.annotation
})
cid_types


['bigg.metabolite',
 'biocyc',
 'chebi',
 'kegg.compound',
 'metanetx.chemical',
 'pubchem.compound',
 'reactome',
 'sbo',
 'seed.compound']

In [7]:
Met_df, Rxn_df = predict_transformed_dG_prime_for_GEM (gem,compartment_conditions=compartment_conditions)

Processing metabolites: 100%|█████| 2748/2748 [00:17<00:00, 153.18it/s]


✅ All molecules exist in the pKa database — skipping prediction.


Predicting transformed standard Gibbs free energy for metabolites: 100%
Predicting transformed standard Gibbs free energy for reactions: 100%|█


In [8]:
Met_df.head()

,bigg.metabolite,chebi,kegg.compound,metanetx.chemical,pubchem.compound
s_0001,"(nan, nan)","(-424.3272705078125, 0.5277105569839478)","(-262.0351257324219, 38.24007797241211)","(nan, nan)",NaN
s_0002,"(nan, nan)","(-410.63547748059545, 0.5277093052864075)","(-250.62523544585773, 38.24007797241211)","(nan, nan)",NaN
s_0003,"(nan, nan)","(-424.3272705078125, 0.5277049541473389)","(-262.0351257324219, 38.24007797241211)","(nan, nan)",NaN
s_0004,"(nan, nan)","(-424.3272705078125, 0.5277089476585388)","(-946.3900756835938, 2.3035659790039062)","(nan, nan)",NaN
s_0006,"(nan, nan)","(-1387.7450104192328, 10.17461109161377)","(-1389.32816768058, 11.83471393585205)","(nan, nan)",NaN


In [9]:
Rxn_df.head()

,dGr_prime,SD of dGr_prime
r_0001,NaN,NaN
r_0002,NaN,NaN
r_0003,15.314423,1.656999
r_0004,NaN,NaN
r_0005,NaN,NaN


In [10]:
Rxn_df.to_csv("./Yeast9_standard_dGr_dGbyG.csv")

In [1]:
import numpy as np
import pandas as pd
import cobra

from ThermoInfer.utils.func import *
from dGbyG.utils.reaction_utils import build_equation

In [2]:
gem_path = "./yeast-GEM.xml"
gem = cobra.io.read_sbml_model(gem_path)

dgr_input_path = "./Yeast9_standard_dGr_dGbyG.csv"
dGr_df = pd.read_csv(dgr_input_path, index_col=0)
dGr = dGr_df[["dGr_prime", "SD of dGr_prime"]].to_numpy()

Set parameter Username


In [3]:
single_compartment_rxn = np.array([
    len(rxn.compartments) == 1
    for rxn in gem.reactions
])

dGr[~single_compartment_rxn, :] = np.nan

In [4]:
tfba_output_path = "./yeast-GEM_Directionality_TFBA.csv"
Directionality_TFBA = pd.read_csv(tfba_output_path, index_col=0)

Directionality_TFBA.head()

,lv,uv,ldGr,udGr
rxn num,,,,
0,-1000.000000,1000.000000,-inf,inf
2,-555.557169,0.000000,-127.209807,81.19389
3,-974.630299,1000.000000,-inf,inf
4,8.122629,59.355756,-inf,inf
5,2.713917,19.831828,-inf,inf


In [5]:
direction_df = pd.DataFrame({
    "built-in label": [
        direction_for_v_range(rxn.lower_bound, rxn.upper_bound) 
        for rxn in gem.reactions
    ]
})

tfba_index = Directionality_TFBA.index.to_numpy()

direction_df.loc[tfba_index, "TFBA v direction"] = [
    direction_for_v_range(row.lv, row.uv)
    for row in Directionality_TFBA.itertuples()
]

direction_df.loc[tfba_index, "TFBA dG direction"] = [
    direction_for_dGr_range(row.ldGr, row.udGr)
    for row in Directionality_TFBA.itertuples()
]

direction_df.loc[:, ["dGr_prime", "SD of dGr_prime"]] = dGr

direction_df.head()

,built-in label,TFBA v direction,TFBA dG direction,dGr_prime,SD of dGr_prime
0,forward,bidirection,bidirection,NaN,NaN
1,forward,nan,nan,NaN,NaN
2,bidirection,backward,bidirection,15.314423,1.656999
3,forward,bidirection,bidirection,NaN,NaN
4,forward,forward,bidirection,NaN,NaN


In [6]:
candidate_mask = (
    direction_df["built-in label"].eq("forward")
    & direction_df["TFBA dG direction"].eq("backward")
    & direction_df["TFBA v direction"].isin(["backward", "blocked"])
)

In [7]:
candidate_data = []

for reaction_number in direction_df.index[candidate_mask]:
    rxn = gem.reactions[int(reaction_number)]
    stoichiometry = {met.name: coeff for met, coeff in rxn.metabolites.items()}
    compartment = sorted(list(rxn.compartments))[0] if len(rxn.compartments) > 0 else ""
    annotation = "; ".join(
        f"{key}: {value}"
        for key, value in rxn.annotation.items()
    )

    candidate_data.append([
        int(reaction_number),
        rxn.id,
        build_equation(stoichiometry, eq_sign="=>"),
        compartment,
        dGr[int(reaction_number)][0],
        dGr[int(reaction_number)][1],
        annotation,
    ])

inconsistent_reaction = pd.DataFrame(
    candidate_data,
    columns=[
        "Reaction number",
        "Reaction ID",
        "Original equation",
        "Compartment",
        "dGr_prime (kJ/mol)",
        "SD of dGr_prime (kJ/mol)",
        "Annotation",
    ]
).set_index("Reaction number", drop=True)

inconsistent_reaction

,Reaction ID,Original equation,Compartment,dGr_prime (kJ/mol),SD of dGr_prime (kJ/mol),Annotation
Reaction number,,,,,,
314,r_0358,"ATP + 2.0 H+ + myo-inositol 1,3,4,5,6-pentakis...",c,48.829168,2.076521,sbo: SBO:0000176; ec-code: 2.7.4.21; kegg.path...
456,r_0573,"1D-myo-inositol 1,4,5,6-tetrakisphosphate + AT...",n,-0.329317,1.768072,sbo: SBO:0000176; ec-code: 2.7.1.151; kegg.pat...
722,r_0953,ammonium + 2.0 H2O + 0.5 oxygen + pyridoxal =>...,c,379.561143,8.063525,sbo: SBO:0000176; ec-code: 1.4.3.5; bigg.react...
764,r_0999,iron(2+) + sirohydrochlorin => 2.0 H+ + siroheme,c,340.780564,61.200075,"sbo: SBO:0000176; ec-code: ['1.3.1.76', '4.99...."
1632,r_2256,4.0 H+ + H2O + trans-oct-2-enoyl-CoA => (R)-3-...,p,-9.781254,1.810396,"sbo: SBO:0000176; ec-code: ['1.1.1.n12', '4.2...."
1639,r_2263,"4.0 H+ + H2O + trans-2,cis-9-octadecadienoyl-C...",p,-22.076168,3.496453,"sbo: SBO:0000176; ec-code: ['1.1.1.n12', '4.2...."
4088,r_4771,ATP + dADP => ADP + dATP,m,-0.000005,0.000243,sbo: SBO:0000176; ec-code: 2.7.4.6; kegg.pathw...


In [8]:
candidate_output_path = "./Yeast9_candidate_inconsistent_reactions.csv"
inconsistent_reaction.to_csv(candidate_output_path)

print(f"Saved candidate reaction table to: {candidate_output_path}")

Saved candidate reaction table to: ./Yeast9_candidate_inconsistent_reactions.csv


In [9]:
gem.reactions.get_by_id('r_4771')

Reaction identifier,r_4771
Name,nucleoside diphosphate kinase
Memory address,0x7a7a9abdedb0
Stoichiometry,s_0437 + s_4319 --> s_0397 + s_4318 ATP + dADP --> ADP + dATP
GPR,YKL067W
Lower bound,0.0
Upper bound,1000.0


In [10]:
import pandas as pd
from IPython.display import display

rxn_id = "r_4771"

# 1. Get reaction object and reaction number
rxn = gem.reactions.get_by_id(rxn_id)
rxn_number = list(gem.reactions).index(rxn)

print("Reaction ID:", rxn.id)
print("Reaction number:", rxn_number)
print("Reaction equation:")
print(rxn.reaction)
print("Compartments:", rxn.compartments)
print("Original bounds:", rxn.lower_bound, rxn.upper_bound)
print("Annotation:")
print(rxn.annotation)

Reaction ID: r_4771
Reaction number: 4088
Reaction equation:
s_0437 + s_4319 --> s_0397 + s_4318
Compartments: {'m'}
Original bounds: 0.0 1000.0
Annotation:
{'sbo': 'SBO:0000176', 'ec-code': '2.7.4.6', 'kegg.pathway': 'Pyrimidine metabolism', 'kegg.reaction': 'R00139', 'metanetx.reaction': 'MNXR106405'}


In [11]:
# Table 4-like result: directionality summary

if rxn_number in direction_df.index:
    table4_row = direction_df.loc[[rxn_number]]
elif str(rxn_number) in direction_df.index:
    table4_row = direction_df.loc[[str(rxn_number)]]
elif rxn_id in direction_df.index:
    table4_row = direction_df.loc[[rxn_id]]
else:
    table4_row = pd.DataFrame()
    print(f"{rxn_id} / reaction number {rxn_number} was not found in direction_df.")

display(table4_row)

,built-in label,TFBA v direction,TFBA dG direction,dGr_prime,SD of dGr_prime
4088,forward,blocked,backward,-0.000005,0.000243


In [12]:
# Table 3-like result: TFBA feasible flux and Gibbs energy bounds

if rxn_number in Directionality_TFBA.index:
    table3_row = Directionality_TFBA.loc[[rxn_number]]
elif str(rxn_number) in Directionality_TFBA.index:
    table3_row = Directionality_TFBA.loc[[str(rxn_number)]]
elif rxn_id in Directionality_TFBA.index:
    table3_row = Directionality_TFBA.loc[[rxn_id]]
else:
    table3_row = pd.DataFrame()
    print(f"{rxn_id} / reaction number {rxn_number} was not found in Directionality_TFBA.")

display(table3_row)

,lv,uv,ldGr,udGr
rxn num,,,,
4088,0.0,0.0,0.0001,96.279857
